# 거래 분류 에이전트 실거래 테스트

가맹점 이름 목록을 **실제 분류 에이전트**(1~6단계, 실제 SVM · 네이버/카카오 검색 · Jev)로 분류하고 결과를 표(pandas DataFrame)로 본다.
임시 회원·거래로 돌린 뒤 DB는 되돌린다. 파일은 따로 만들지 않는다.

- 실거래 이름은 개인 데이터라 `local_workspace/data/classifier/real_merchants.txt`(Git 제외)에 둔다. **노트북 출력은 지우고 커밋**
- 노트북은 `scripts/` 폴더에서 열고 커널은 저장소의 `.venv`. Docker DB가 켜져 있어야 한다
- 5단계까지 가는 이름은 네이버·Jev를 실제로 호출한다 (상호만 전송). `.env` 키가 없으면 4단계에서 끝난다

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv

ROOT = Path(find_dotenv(usecwd=True)).parent              # 저장소 루트 (.env 가 있는 곳)
sys.path.insert(0, str(ROOT / "backend"))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"          # 주피터는 이벤트 루프 안에서 돌아서 Django DB 호출에 필요
import django
django.setup()
from finance.management.commands.classify_names import classify_names
from scripts.spending_classifier.common import CATEGORIES

NAMES = ROOT / "local_workspace/data/classifier/real_merchants.txt"   # 한 줄에 가맹점 이름 하나
names = [s.strip() for s in NAMES.read_text(encoding="utf-8-sig").splitlines() if s.strip()]
NAME_OF = {**CATEGORIES, "ETC": "기타", "UNCLASSIFIED": "확인 필요"}
pd.set_option("display.max_rows", 200)
len(names)

## 1. 실제 분류 실행 → 결과 표
약 30초~1분. 단계: `1 다품목` · `1 페이 충전` · `3 규칙` · `3 이름 모름` · `4 SVM` · `5 검색+Jev`. 결과가 `확인 필요`면 사용자에게 묻는 거래

In [ ]:
df = pd.DataFrame(classify_names(names))
df["결과"] = df["결과"].map(lambda c: NAME_OF.get(c, c))
df["후보"] = df["후보"].map(lambda s: " ".join(NAME_OF.get(c, c) for c in s.split()))
df[["이름", "결과", "단계", "확신도", "후보", "검색", "Jev", "ms", "메모"]]

## 2. 단계별 · 결과별 요약

In [ ]:
print(f"총 {len(df)}건 · 자동 분류 {(df['결과'] != '확인 필요').sum()}건 · 확인 필요 {(df['결과'] == '확인 필요').sum()}건")
pd.crosstab(df["단계"], df["결과"], margins=True, margins_name="합계")

## 3. 확인 필요 거래와 이유
`메모`: `store_mismatch` = 검색 첫 결과가 다른 가게, `dropped_industry` = 가게가 아닌 업종(금융 등)이라 근거에서 뺌

In [ ]:
df[df["결과"] == "확인 필요"][["이름", "단계", "후보", "검색", "Jev", "메모", "이유"]]

## 4. 이름 몇 개만 바로 시험해 보기

In [ ]:
pd.DataFrame(classify_names(["스타벅스 강남점", "비식별거래처_0001", "카카오T 택시"]))